In [30]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

class EnhancedBollingerBandStrategy:
    def __init__(self, symbol="SPY", start_date="2011-01-01", end_date="2021-12-31", 
                 train_split_date="2019-01-01", initial_balance=1000, transaction_fee=1.0):
        self.symbol = symbol
        self.start_date = start_date
        self.end_date = end_date
        self.train_split_date = train_split_date
        self.initial_balance = initial_balance
        self.transaction_fee = transaction_fee  # Fixed transaction fee per trade
        self.data = None
        self.train_data = None
        self.test_data = None
        self.model = None
        self.predictions = None
        
    # ================ STEP 1: DATA PREPARATION ================
    def download_and_prepare_data(self):
        """
        Step 1: Download data and create all necessary technical indicators
        按論文3.1節的要求準備數據
        """
        print("Step 1: Downloading and preparing data...")
        
        # Download data
        self.data = yf.download(self.symbol, self.start_date, self.end_date)
        
        # Calculate WMA(3) - 3-day weighted moving average
        self.data["WMA"] = (3*self.data.Close + 2*self.data.Close.shift(1) + self.data.Close.shift(2)) / 6
        
        # Create historical WMA features (past 5 days)
        for i in range(1, 6):
            self.data[f"WMA_lag_{i}"] = self.data["WMA"].shift(i)
        
        # Calculate Bollinger Bands components (d=20, k=3)
        self.data["MA"] = self.data.Close.rolling(20).mean()
        self.data["std"] = self.data.Close.rolling(window=20).std()
        self.data["Upper_Track"] = self.data.MA + 1.7 * self.data["std"]
        self.data["Lower_Track"] = self.data.MA - 1.7 * self.data["std"]
        
        # Calculate ATR (Average True Range)
        self.data["high_low"] = self.data.High - self.data.Low
        self.data["high_close"] = (self.data.High - self.data.Close.shift(1)).abs()
        self.data["low_close"] = (self.data.Low - self.data.Close.shift(1)).abs()
        self.data["TR"] = pd.concat([self.data.high_low, self.data.high_close, 
                                   self.data.low_close], axis=1).max(axis=1)
        self.data["ATR"] = self.data.TR.rolling(20).mean()
        
        # Create target variable: WMA(t+1) - WMA(t)
        self.data["WMA_next"] = self.data["WMA"].shift(-1)
        self.data["WMA_diff"] = self.data["WMA_next"] - self.data["WMA"]
        
        # Remove rows with NaN values
        self.data = self.data.dropna()
        
        # Split into train and test sets
        self.train_data = self.data[self.data.index < self.train_split_date]
        self.test_data = self.data[self.data.index >= self.train_split_date]
        
        print(f"Data prepared successfully!")
        print(f"Training set: {len(self.train_data)} samples")
        print(f"Testing set: {len(self.test_data)} samples")
        
        return self.data
    
    # ================ STEP 2: FEATURE ENGINEERING ================
    def prepare_features(self):
        """
        Step 2: Prepare features for Random Forest
        按論文3.2節準備特徵變數
        """
        print("\nStep 2: Preparing features for Random Forest...")
        
        # Feature columns (11 features as mentioned in paper)
        feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 
                       'WMA_lag_1', 'WMA_lag_2', 'WMA_lag_3', 'WMA_lag_4', 'WMA_lag_5',
                       'WMA']
        
        self.X_train = self.train_data[feature_cols]
        self.y_train = self.train_data['WMA_diff']
        
        self.X_test = self.test_data[feature_cols]
        self.y_test = self.test_data['WMA_diff']
        
        print(f"Features prepared: {self.X_train.shape[1]} features")
        print(f"Feature columns: {feature_cols}")
        
        return self.X_train, self.y_train, self.X_test, self.y_test
    
    # ================ STEP 3: RANDOM FOREST TRAINING ================
    def train_random_forest(self, n_estimators=100, random_state=42):
        """
        Step 3: Train Random Forest model
        按論文3.2節訓練Random Forest模型
        """
        print("\nStep 3: Training Random Forest model...")
        
        self.model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)
        self.model.fit(self.X_train, self.y_train)
        
        # Make predictions
        train_pred = self.model.predict(self.X_train)
        test_pred = self.model.predict(self.X_test)
        
        # Calculate predicted WMA values
        self.test_data = self.test_data.copy()
        self.test_data['WMA_diff_pred'] = test_pred
        self.test_data['WMA_next_pred'] = self.test_data['WMA'] + self.test_data['WMA_diff_pred']
        
        print("Random Forest training completed!")
        return self.model
    
    # ================ STEP 4: MODEL EVALUATION ================
    def evaluate_model(self):
        """
        Step 4: Evaluate Random Forest performance
        按論文4.1節評估模型性能
        """
        print("\nStep 4: Evaluating model performance...")
        
        # Get predictions
        test_pred = self.test_data['WMA_diff_pred']
        actual = self.test_data['WMA_diff']
        
        # Calculate metrics
        r2 = r2_score(actual, test_pred)
        rmse = np.sqrt(mean_squared_error(actual, test_pred))
        mae = mean_absolute_error(actual, test_pred)
        
        print(f"Model Performance:")
        print(f"R² Score: {r2:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
        
        # Plot actual vs predicted
        plt.figure(figsize=(12, 6))
        plt.plot(self.test_data.index, self.test_data['WMA_next'], label='Actual WMA', alpha=0.7)
        plt.plot(self.test_data.index, self.test_data['WMA_next_pred'], label='Predicted WMA', alpha=0.7)
        plt.title(f'{self.symbol} - Actual vs Predicted WMA')
        plt.xlabel('Date')
        plt.ylabel('WMA Value')
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
        return {'r2': r2, 'rmse': rmse, 'mae': mae}
    
    # ================ STEP 5: TRADITIONAL BOLLINGER BAND STRATEGY ================
    def traditional_bollinger_strategy(self):
        """
        Step 5: Implement Traditional Bollinger Band Strategy (Algorithm 1)
        按論文Algorithm 1實現傳統布林通道策略
        """
        print("\nStep 5: Running Traditional Bollinger Band Strategy...")
        
        test_data = self.test_data.copy()
        
        # Initialize trading variables
        position = 0  # 0: no position, 1: long position
        balance = self.initial_balance
        shares = 0
        trades = []
        portfolio_values = []
        total_fees_paid = 0  # Track total fees paid
        
        for i, (date, row) in enumerate(test_data.iterrows()):
            # Convert to scalar values to avoid pandas comparison issues
            close_price = float(row['Close'])
            upper_track = float(row['Upper_Track'])
            lower_track = float(row['Lower_Track'])
            
            # Traditional Bollinger Band Logic
            if position == 0 and close_price <= lower_track:
                # Open long position (check if we have enough balance for transaction fee)
                if balance > self.transaction_fee:
                    available_balance = balance - self.transaction_fee
                    shares = available_balance / close_price
                    balance = 0
                    position = 1
                    total_fees_paid += self.transaction_fee
                    trades.append({
                        'date': date, 
                        'action': 'BUY', 
                        'price': close_price, 
                        'shares': shares,
                        'fee': self.transaction_fee
                    })
                
            elif position == 1 and close_price >= upper_track:
                # Close long position (deduct transaction fee from proceeds)
                gross_proceeds = shares * close_price
                balance = gross_proceeds - self.transaction_fee
                position = 0
                total_fees_paid += self.transaction_fee
                trades.append({
                    'date': date, 
                    'action': 'SELL', 
                    'price': close_price, 
                    'shares': shares,
                    'fee': self.transaction_fee,
                    'gross_proceeds': gross_proceeds,
                    'net_proceeds': balance
                })
                shares = 0
            
            # Calculate portfolio value
            if position == 1:
                portfolio_value = shares * close_price
            else:
                portfolio_value = balance
            
            portfolio_values.append(portfolio_value)
        
        # Final portfolio value
        final_value = portfolio_values[-1]
        model_return = (final_value - self.initial_balance) / self.initial_balance * 100
        
        # Buy and hold return
        start_price = float(test_data.iloc[0]['Close'])
        end_price = float(test_data.iloc[-1]['Close'])
        basic_return = (end_price - start_price) / start_price * 100
        
        # Calculate max drawdown
        portfolio_series = pd.Series(portfolio_values, index=test_data.index)
        running_max = portfolio_series.expanding().max()
        drawdown = (portfolio_series - running_max) / running_max * 100
        max_drawdown = drawdown.min()
        
        print(f"Traditional Bollinger Band Strategy Results:")
        print(f"Model Return: {model_return:.2f}%")
        print(f"Basic Return: {basic_return:.2f}%")
        print(f"Max Drawdown: {max_drawdown:.2f}%")
        print(f"Number of trades: {len(trades)}")
        print(f"Total fees paid: ${total_fees_paid:.2f}")
        print(f"Fee impact on return: {(total_fees_paid / self.initial_balance * 100):.2f}%")
        
        return {
            'model_return': model_return,
            'basic_return': basic_return,
            'max_drawdown': max_drawdown,
            'trades': trades,
            'portfolio_values': portfolio_values,
            'total_fees_paid': total_fees_paid
        }
    
    # ================ STEP 6: ENHANCED BOLLINGER BAND STRATEGY ================
    def enhanced_bollinger_strategy(self):
        """
        Step 6: Implement Enhanced Bollinger Band Strategy (Algorithm 2)
        按論文Algorithm 2實現增強版布林通道策略
        """
        print("\nStep 6: Running Enhanced Bollinger Band Strategy...")
        
        test_data = self.test_data.copy()
        
        # Initialize trading variables
        position = 0  # 0: no position, 1: long position, -1: short position
        balance = self.initial_balance
        shares = 0
        entry_price = 0
        trades = []
        portfolio_values = []
        total_fees_paid = 0  # Track total fees paid
        
        for i, (date, row) in enumerate(test_data.iterrows()):
            # Convert to scalar values to avoid pandas comparison issues
            close_price = float(row['Close'])
            upper_track = float(row['Upper_Track'])
            lower_track = float(row['Lower_Track'])
            predicted_wma = float(row['WMA_next_pred'])
            atr = float(row['ATR'])
            
            # Enhanced Bollinger Band Logic using predicted WMA
            if position == 0 and predicted_wma <= lower_track:
                # Open long position (check if we have enough balance for transaction fee)
                if balance > self.transaction_fee:
                    available_balance = balance - self.transaction_fee
                    shares = available_balance / close_price
                    balance = 0
                    position = 1
                    entry_price = close_price
                    total_fees_paid += self.transaction_fee
                    trades.append({
                        'date': date, 
                        'action': 'BUY', 
                        'price': close_price, 
                        'shares': shares,
                        'fee': self.transaction_fee
                    })
                
            elif position == 0 and predicted_wma >= upper_track:
                # Open short position (for simplicity, we'll skip short positions in this implementation)
                # In a real implementation, you would need to handle margin requirements
                pass
                
            elif position == 1:  # Long position
                # Close conditions for long position
                if predicted_wma >= upper_track or close_price <= (entry_price - 1.8 * atr):
                    gross_proceeds = shares * close_price
                    balance = gross_proceeds - self.transaction_fee
                    position = 0
                    total_fees_paid += self.transaction_fee
                    trades.append({
                        'date': date, 
                        'action': 'SELL', 
                        'price': close_price, 
                        'shares': shares,
                        'fee': self.transaction_fee,
                        'gross_proceeds': gross_proceeds,
                        'net_proceeds': balance
                    })
                    shares = 0
            
            # Calculate portfolio value
            if position == 1:  # Long position
                portfolio_value = shares * close_price
            else:
                portfolio_value = balance
            
            portfolio_values.append(portfolio_value)
        
        # Final portfolio value (close any remaining position)
        if position == 1:
            final_close_price = float(test_data.iloc[-1]['Close'])
            gross_proceeds = shares * final_close_price
            balance = gross_proceeds - self.transaction_fee
            total_fees_paid += self.transaction_fee
            trades.append({
                'date': test_data.index[-1], 
                'action': 'SELL', 
                'price': final_close_price, 
                'shares': shares,
                'fee': self.transaction_fee,
                'gross_proceeds': gross_proceeds,
                'net_proceeds': balance
            })
        
        final_value = portfolio_values[-1] if portfolio_values else balance
        model_return = (final_value - self.initial_balance) / self.initial_balance * 100
        
        # Buy and hold return
        start_price = float(test_data.iloc[0]['Close'])
        end_price = float(test_data.iloc[-1]['Close'])
        basic_return = (end_price - start_price) / start_price * 100
        
        # Calculate max drawdown
        portfolio_series = pd.Series(portfolio_values, index=test_data.index)
        running_max = portfolio_series.expanding().max()
        drawdown = (portfolio_series - running_max) / running_max * 100
        max_drawdown = drawdown.min()
        
        # Calculate Sharpe Ratio (simplified)
        returns = portfolio_series.pct_change().dropna()
        sharpe_ratio = returns.mean() / returns.std() * np.sqrt(252) if returns.std() > 0 else 0
        
        print(f"Enhanced Bollinger Band Strategy Results:")
        print(f"Model Return: {model_return:.2f}%")
        print(f"Basic Return: {basic_return:.2f}%")
        print(f"Max Drawdown: {max_drawdown:.2f}%")
        print(f"Sharpe Ratio: {sharpe_ratio:.4f}")
        print(f"Number of trades: {len(trades)}")
        print(f"Total fees paid: ${total_fees_paid:.2f}")
        print(f"Fee impact on return: {(total_fees_paid / self.initial_balance * 100):.2f}%")
        
        return {
            'model_return': model_return,
            'basic_return': basic_return,
            'max_drawdown': max_drawdown,
            'sharpe_ratio': sharpe_ratio,
            'trades': trades,
            'portfolio_values': portfolio_values,
            'total_fees_paid': total_fees_paid
        }
    
    # ================ STEP 7: VISUALIZATION AND COMPARISON ================
    def plot_strategy_comparison(self, traditional_results, enhanced_results):
        """
        Step 7: Visualize and compare strategy performance
        可視化策略表現比較
        """
        print("\nStep 7: Visualizing strategy comparison...")
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))
        
        # Plot 1: Price and Bollinger Bands with signals
        test_data = self.test_data
        ax1.plot(test_data.index, test_data['Close'], label='Close Price', linewidth=1)
        ax1.plot(test_data.index, test_data['Upper_Track'], label='Upper Track', alpha=0.7)
        ax1.plot(test_data.index, test_data['Lower_Track'], label='Lower Track', alpha=0.7)
        ax1.fill_between(test_data.index, test_data['Upper_Track'], test_data['Lower_Track'], alpha=0.1)
        
        # Add trade signals for enhanced strategy
        enhanced_trades = enhanced_results['trades']
        for trade in enhanced_trades:
            if trade['action'] == 'BUY':
                ax1.scatter(trade['date'], trade['price'], color='green', marker='^', s=100, zorder=5)
            elif trade['action'] == 'SELL':
                ax1.scatter(trade['date'], trade['price'], color='red', marker='v', s=100, zorder=5)
        
        ax1.set_title(f'{self.symbol} - Price with Enhanced Bollinger Band Signals')
        ax1.set_ylabel('Price')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Portfolio Performance Comparison
        traditional_portfolio = pd.Series(traditional_results['portfolio_values'], index=test_data.index)
        enhanced_portfolio = pd.Series(enhanced_results['portfolio_values'], index=test_data.index)
        
        ax2.plot(test_data.index, traditional_portfolio, label='Traditional Bollinger Band', linewidth=2)
        ax2.plot(test_data.index, enhanced_portfolio, label='Enhanced Bollinger Band', linewidth=2)
        ax2.axhline(y=self.initial_balance, color='black', linestyle='--', alpha=0.5, label='Initial Balance')
        
        ax2.set_title('Portfolio Value Comparison')
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Portfolio Value')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    # ================ MAIN EXECUTION FUNCTION ================
    def run_complete_strategy(self):
        """
        Execute the complete Enhanced Bollinger Band Strategy
        執行完整的增強版布林通道策略
        """
        print("=" * 60)
        print("ENHANCED BOLLINGER BAND STRATEGY - COMPLETE IMPLEMENTATION")
        print("=" * 60)
        
        # Step 1: Data Preparation
        self.download_and_prepare_data()
        
        # Step 2: Feature Engineering
        self.prepare_features()
        
        # Step 3: Train Random Forest
        self.train_random_forest()
        
        # Step 4: Evaluate Model
        model_metrics = self.evaluate_model()
        
        # Step 5: Traditional Strategy
        traditional_results = self.traditional_bollinger_strategy()
        
        # Step 6: Enhanced Strategy
        enhanced_results = self.enhanced_bollinger_strategy()
        
        # Step 7: Comparison and Visualization
        self.plot_strategy_comparison(traditional_results, enhanced_results)
        
        # Final Summary
        print("\n" + "=" * 60)
        print("FINAL STRATEGY COMPARISON SUMMARY")
        print("=" * 60)
        print(f"Traditional Strategy Return: {traditional_results['model_return']:.2f}%")
        print(f"Enhanced Strategy Return: {enhanced_results['model_return']:.2f}%")
        print(f"Basic Buy & Hold Return: {enhanced_results['basic_return']:.2f}%")
        improvement = enhanced_results['model_return'] - traditional_results['model_return']
        print(f"Enhancement Improvement: {improvement:.2f}%")
        print(f"Traditional Strategy Fees: ${traditional_results['total_fees_paid']:.2f}")
        print(f"Enhanced Strategy Fees: ${enhanced_results['total_fees_paid']:.2f}")
        
        return {
            'model_metrics': model_metrics,
            'traditional_results': traditional_results,
            'enhanced_results': enhanced_results
        }

# ================ USAGE EXAMPLE ================
if __name__ == "__main__":
    # Create strategy instance
    strategy = EnhancedBollingerBandStrategy(
        symbol="BTC", 
        start_date="2015-08-10", 
        end_date="2025-08-10",
        train_split_date="2023-08-10",
        transaction_fee=1.0  # $5 per transaction
    )
    
    # Run complete strategy
    results = strategy.run_complete_strategy()

ENHANCED BOLLINGER BAND STRATEGY - COMPLETE IMPLEMENTATION
Step 1: Downloading and preparing data...


[*********************100%***********************]  1 of 1 completed


Data prepared successfully!
Training set: 0 samples
Testing set: 237 samples

Step 2: Preparing features for Random Forest...
Features prepared: 11 features
Feature columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'WMA_lag_1', 'WMA_lag_2', 'WMA_lag_3', 'WMA_lag_4', 'WMA_lag_5', 'WMA']

Step 3: Training Random Forest model...


ValueError: Found array with 0 sample(s) (shape=(0, 11)) while a minimum of 1 is required by RandomForestRegressor.